In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import glob

BASE_PATH = '/content/drive/MyDrive'

found = glob.glob(f'{BASE_PATH}/**/procedures_icd.csv*', recursive=True)
print("procedures_icd files found:", found)
found_icu = glob.glob(f'{BASE_PATH}/**/icustays.csv*', recursive=True)
print("icustays files found:", found_icu)
found_dict = glob.glob(f'{BASE_PATH}/**/d_icd_procedures.csv*', recursive=True)
print("d_icd_procedures files found:", found_dict)

In [ ]:
PROCEDURES_PATH = '/content/drive/MyDrive/mimic-iv/hosp/procedures_icd.csv.gz'
ICUSTAYS_PATH = '/content/drive/MyDrive/mimic-iv/icu/icustays.csv.gz'
DICT_PATH = '/content/drive/MyDrive/mimic-iv/hosp/d_icd_procedures.csv.gz'

In [ ]:
import pandas as pd

proc = pd.read_csv(PROCEDURES_PATH, dtype={'icd_code': str})
icu = pd.read_csv(ICUSTAYS_PATH)

In [ ]:
proc_icu = proc.merge(icu[['subject_id', 'hadm_id', 'stay_id']], on=['subject_id', 'hadm_id'], how='inner')

code_counts = proc_icu.groupby(['icd_code', 'icd_version']).agg(
    n_admissions=('hadm_id', 'nunique'),
    n_patients=('subject_id', 'nunique'),
    n_icu_stays=('stay_id', 'nunique')
).reset_index()

d_icd = pd.read_csv(DICT_PATH, dtype={'icd_code': str})
code_counts = code_counts.merge(d_icd[['icd_code', 'icd_version', 'long_title']], on=['icd_code', 'icd_version'], how='left')
code_counts = code_counts.sort_values('n_admissions', ascending=False)

code_counts.to_csv('/content/drive/MyDrive/mimic-iv/icu_procedure_volume_ranked.csv', index=False)
print(code_counts.head(50).to_string(index=False))

In [ ]:
mask_vessel = d_icd['long_title'].str.lower().str.contains('aorta', na=False)
mask_op = d_icd['long_title'].str.lower().str.contains(
    'resection.*replacement|restriction|endovascular implantation',
    regex=True, na=False
)
aorta_candidates = d_icd[mask_vessel & mask_op].sort_values(['icd_version', 'icd_code'])
print(aorta_candidates[['icd_code', 'icd_version', 'long_title']].to_string(index=False))

In [ ]:
mask_vessel2 = d_icd['long_title'].str.lower().str.contains('aorta', na=False)
mask_op2 = d_icd['long_title'].str.lower().str.contains(
    'replacement|restriction|endovascular implantation',
    regex=True, na=False
)
mask_exclude = d_icd['long_title'].str.lower().str.contains('valve|ascending/arch', regex=True, na=False)
aorta_all = d_icd[mask_vessel2 & mask_op2 & ~mask_exclude]

usage = proc[proc['icd_code'].isin(aorta_all['icd_code'])].groupby(['icd_code', 'icd_version']).agg(
    n_admissions=('hadm_id', 'nunique')
).reset_index()
usage = usage.merge(aorta_all[['icd_code', 'icd_version', 'long_title']], on=['icd_code', 'icd_version'])
usage = usage.sort_values('n_admissions', ascending=False)
print(usage.to_string(index=False))

In [ ]:
def classify_location(title):
    t = title.lower()
    if 'thoracic aorta' in t:
        return 'thoracic'
    elif 'abdominal aorta' in t:
        return 'abdominal'
    return 'other'

def classify_approach(title):
    t = title.lower()
    if 'restriction' in t or 'endovascular implantation' in t:
        return 'endovascular'
    elif 'replacement' in t or 'resection' in t:
        return 'open'
    return 'other'

aorta_icu = proc_icu[proc_icu['icd_code'].isin(aorta_all['icd_code'])].copy()
aorta_icu = aorta_icu.merge(aorta_all[['icd_code', 'icd_version', 'long_title']], on=['icd_code', 'icd_version'])
aorta_icu['location'] = aorta_icu['long_title'].apply(classify_location)
aorta_icu['approach'] = aorta_icu['long_title'].apply(classify_approach)

# dedupe first — a single admission can have multiple codes (e.g. a bifurcation
# modifier code alongside the main code), so count distinct admissions, not code rows
summary = aorta_icu.drop_duplicates(subset=['hadm_id', 'location', 'approach']).groupby(['location', 'approach']).agg(
    n_admissions=('hadm_id', 'nunique'),
    n_patients=('subject_id', 'nunique'),
    n_icu_stays=('stay_id', 'nunique')
).reset_index()
print(summary.to_string(index=False))

# and the true combined total (a patient counted once even if flagged in both
# location/approach groups somehow)
total = aorta_icu[['subject_id', 'hadm_id', 'stay_id']].drop_duplicates()
print(f"\nTotal combined: {total['hadm_id'].nunique()} admissions, {total['subject_id'].nunique()} patients, {total['stay_id'].nunique()} ICU stays")

In [ ]:
import pandas as pd

def classify_location(title):
    t = title.lower()
    if 'thoracic' in t:
        return 'thoracic'
    elif 'abdominal' in t:
        return 'abdominal'
    return 'other'

def classify_approach(title):
    t = title.lower()
    if 'restriction' in t or 'endovascular implantation' in t:
        return 'endovascular'
    elif 'replacement' in t or 'resection' in t:
        return 'open'
    return 'other'

proc_icu = proc.merge(icu[['subject_id', 'hadm_id', 'stay_id']], on=['subject_id', 'hadm_id'], how='inner')

d_icd = pd.read_csv(DICT_PATH, dtype={'icd_code': str})
mask_vessel2 = d_icd['long_title'].str.lower().str.contains('aorta', na=False)
mask_op2 = d_icd['long_title'].str.lower().str.contains(
    'replacement|restriction|endovascular implantation',
    regex=True, na=False
)
mask_exclude = d_icd['long_title'].str.lower().str.contains('valve|ascending/arch', regex=True, na=False)
aorta_all = d_icd[mask_vessel2 & mask_op2 & ~mask_exclude]

aorta_icu = proc_icu[proc_icu['icd_code'].isin(aorta_all['icd_code'])].copy()
aorta_icu = aorta_icu.merge(aorta_all[['icd_code', 'icd_version', 'long_title']], on=['icd_code', 'icd_version'])
aorta_icu['location'] = aorta_icu['long_title'].apply(classify_location)
aorta_icu['approach'] = aorta_icu['long_title'].apply(classify_approach)

summary = aorta_icu.drop_duplicates(subset=['hadm_id', 'location', 'approach']).groupby(['location', 'approach']).agg(
    n_admissions=('hadm_id', 'nunique'),
    n_patients=('subject_id', 'nunique'),
    n_icu_stays=('stay_id', 'nunique')
).reset_index()
print(summary.to_string(index=False))

total = aorta_icu[['subject_id', 'hadm_id', 'stay_id']].drop_duplicates()
print(f"\nTotal combined: {total['hadm_id'].nunique()} admissions, {total['subject_id'].nunique()} patients, {total['stay_id'].nunique()} ICU stays")

In [ ]:
dup_check = aorta_icu.drop_duplicates(subset=['hadm_id','location','approach']).groupby('hadm_id').size()
overlap = dup_check[dup_check > 1]
print(f"Admissions spanning multiple location/approach categories: {len(overlap)}")

In [ ]:
dedup = aorta_icu.drop_duplicates(subset=['hadm_id','location','approach'])
overlap_hadm = dup_check[dup_check > 1].index
overlap_detail = dedup[dedup['hadm_id'].isin(overlap_hadm)].groupby('hadm_id').agg(
    locations=('location', lambda x: tuple(sorted(set(x)))),
    approaches=('approach', lambda x: tuple(sorted(set(x))))
).reset_index()
print("Location combos:")
print(overlap_detail['locations'].value_counts())
print("\nApproach combos:")
print(overlap_detail['approaches'].value_counts())

In [ ]:
def relabel(row):
    locs = row['locations']
    apps = row['approaches']
    if len(apps) > 1:
        return 'conversion'  # flag for exclusion
    if len(locs) > 1 and 'other' not in locs:
        return 'thoracoabdominal'
    return None  # not an overlap case, keep original classification

overlap_detail['new_location'] = overlap_detail.apply(relabel, axis=1)
print(overlap_detail[['hadm_id', 'new_location']].value_counts('new_location'))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
for p in ['/content/drive/MyDrive/mimic-iv/hosp/procedures_icd.csv.gz',
          '/content/drive/MyDrive/mimic-iv/icu/icustays.csv.gz',
          '/content/drive/MyDrive/mimic-iv/hosp/d_icd_procedures.csv.gz']:
    print(os.path.exists(p), p)

In [ ]:
import pandas as pd

# Correct paths using the subdirectories identified in the file search
PROCEDURES_PATH = '/content/drive/MyDrive/mimic-iv/hosp/procedures_icd.csv.gz'
ICUSTAYS_PATH = '/content/drive/MyDrive/mimic-iv/icu/icustays.csv.gz'
DICT_PATH = '/content/drive/MyDrive/mimic-iv/hosp/d_icd_procedures.csv.gz'

# Recreate aorta_icu using the definitions established in earlier cells
def classify_location(title):
    t = title.lower()
    if 'thoracic' in t:
        return 'thoracic'
    elif 'abdominal' in t:
        return 'abdominal'
    return 'other'

def classify_approach(title):
    t = title.lower()
    if 'restriction' in t or 'endovascular implantation' in t:
        return 'endovascular'
    elif 'replacement' in t or 'resection' in t:
        return 'open'
    return 'other'

# Read necessary tables
proc = pd.read_csv(PROCEDURES_PATH, dtype={'icd_code': str})
icu = pd.read_csv(ICUSTAYS_PATH)
d_icd = pd.read_csv(DICT_PATH, dtype={'icd_code': str})

proc_icu = proc.merge(icu[['subject_id', 'hadm_id', 'stay_id']], on=['subject_id', 'hadm_id'], how='inner')
mask_vessel2 = d_icd['long_title'].str.lower().str.contains('aorta', na=False)
mask_op2 = d_icd['long_title'].str.lower().str.contains('replacement|restriction|endovascular implantation', regex=True, na=False)
mask_exclude = d_icd['long_title'].str.lower().str.contains('valve|ascending/arch', regex=True, na=False)
aorta_all = d_icd[mask_vessel2 & mask_op2 & ~mask_exclude]

aorta_icu = proc_icu[proc_icu['icd_code'].isin(aorta_all['icd_code'])].copy()
aorta_icu = aorta_icu.merge(aorta_all[['icd_code', 'icd_version', 'long_title']], on=['icd_code', 'icd_version'])
aorta_icu['location'] = aorta_icu['long_title'].apply(classify_location)
aorta_icu['approach'] = aorta_icu['long_title'].apply(classify_approach)

# Recreate the missing variables from the preceding cells to make this cell self-contained
dup_check = aorta_icu.drop_duplicates(subset=['hadm_id','location','approach']).groupby('hadm_id').size()
dedup = aorta_icu.drop_duplicates(subset=['hadm_id','location','approach'])
overlap_hadm = dup_check[dup_check > 1].index
overlap_detail = dedup[dedup['hadm_id'].isin(overlap_hadm)].groupby('hadm_id').agg(
    locations=('location', lambda x: tuple(sorted(set(x)))),
    approaches=('approach', lambda x: tuple(sorted(set(x))))
).reset_index()

def relabel(row):
    locs = row['locations']
    apps = row['approaches']
    if len(apps) > 1:
        return 'conversion'  # flag for exclusion
    if len(locs) > 1 and 'other' not in locs:
        return 'thoracoabdominal'
    return None  # not an overlap case, keep original classification

overlap_detail['new_location'] = overlap_detail.apply(relabel, axis=1)

conversion_hadm = overlap_detail.loc[overlap_detail['new_location'] == 'conversion', 'hadm_id']
thoracoabd_hadm = overlap_detail.loc[overlap_detail['new_location'] == 'thoracoabdominal', 'hadm_id']

print("Rows in aorta_icu (before dedupe):", len(aorta_icu), "| unique admissions:", aorta_icu['hadm_id'].nunique())
cohort = aorta_icu.drop_duplicates(subset=['hadm_id']).copy()
print("A. admissions with ICU stay + aortic repair code:", len(cohort))
cohort = cohort[~cohort['hadm_id'].isin(conversion_hadm)]
print("B. after dropping conversions:", len(cohort))
cohort.loc[cohort['hadm_id'].isin(thoracoabd_hadm), 'location'] = 'thoracoabdominal'
print("   'other' location admissions about to be dropped:", int((cohort['location'] == 'other').sum()))
cohort = cohort[cohort['location'] != 'other']
print("C. after dropping 'other':", len(cohort))

cohort = cohort[['subject_id', 'hadm_id', 'stay_id', 'location', 'approach']].drop_duplicates(subset='hadm_id')
print(f"Final cohort: {len(cohort)} admissions")
print("Unique patients:", cohort['subject_id'].nunique())

In [ ]:
other_hadm = aorta_icu[aorta_icu['location'] == 'other']['hadm_id'].unique()
print(len(other_hadm), "ambiguous admissions")
print(aorta_icu[aorta_icu['hadm_id'].isin(other_hadm)].icd_code.value_counts().head(10).to_string())

In [ ]:
admissions_path = glob.glob(f'{BASE_PATH}/**/admissions.csv*', recursive=True)
print(admissions_path)

In [ ]:
admissions_path = glob.glob(f'{BASE_PATH}/**/admissions.csv*', recursive=True)
print(admissions_path)

In [ ]:
ADMISSIONS_PATH = admissions_path[0]
admissions = pd.read_csv(ADMISSIONS_PATH, usecols=['subject_id', 'hadm_id', 'admittime', 'dischtime'], parse_dates=['admittime', 'dischtime'])
cohort = cohort.merge(admissions, on=['subject_id', 'hadm_id'], how='left')
cohort['admit_year'] = cohort['admittime'].dt.year
print(cohort['admit_year'].describe())

In [ ]:
patients_path = glob.glob(f'{BASE_PATH}/**/patients.csv*', recursive=True)
print(patients_path)

In [ ]:
PATIENTS_PATH = patients_path[0]
patients = pd.read_csv(PATIENTS_PATH, usecols=['subject_id', 'anchor_year', 'anchor_year_group'])
cohort = cohort.merge(patients, on='subject_id', how='left')
print(cohort['anchor_year_group'].value_counts().sort_index())

In [ ]:
labitems_path = glob.glob(f'{BASE_PATH}/**/d_labitems.csv*', recursive=True)
labevents_path = glob.glob(f'{BASE_PATH}/**/labevents.csv*', recursive=True)
print(labitems_path, labevents_path)

In [ ]:
d_labitems = pd.read_csv(labitems_path[0])
creat_items = d_labitems[d_labitems['label'].str.contains('creatinine', case=False, na=False)]
print(creat_items[['itemid', 'label', 'fluid']])

In [ ]:
CREAT_ITEMID = 50912

cohort_subjects = set(cohort['subject_id'])
chunks = []
for chunk in pd.read_csv(labevents_path[0], usecols=['subject_id', 'hadm_id', 'itemid', 'charttime', 'valuenum'],
                          parse_dates=['charttime'], chunksize=2_000_000):
    filtered = chunk[(chunk['itemid'] == CREAT_ITEMID) & (chunk['subject_id'].isin(cohort_subjects))]
    if len(filtered):
        chunks.append(filtered)

creat = pd.concat(chunks, ignore_index=True)
print(f"Creatinine values pulled: {len(creat)}")
print(f"Admissions with at least one value: {creat['hadm_id'].nunique()} / {len(cohort)}")

In [ ]:
creat = creat.merge(cohort[['subject_id', 'hadm_id', 'admittime', 'dischtime']], on=['subject_id', 'hadm_id'], how='inner')
print(f"Admissions with creatinine data (correctly restricted to the index admission): {creat['hadm_id'].nunique()} / {len(cohort)}")

baseline_window = creat[creat['charttime'] <= creat['admittime'] + pd.Timedelta(hours=24)]
baseline = baseline_window.groupby('hadm_id')['valuenum'].min().rename('baseline_creat')

peak = creat.groupby('hadm_id')['valuenum'].max().rename('peak_creat')

aki = pd.concat([baseline, peak], axis=1).reset_index()
aki['aki'] = (aki['peak_creat'] >= 1.5 * aki['baseline_creat']).astype(int)

cohort_final = cohort.merge(aki, on='hadm_id', how='left')
print(f"Admissions with usable creatinine data: {cohort_final['aki'].notna().sum()} / {len(cohort_final)}")
print(cohort_final['aki'].value_counts(dropna=False))
print(f"AKI rate: {cohort_final['aki'].mean():.1%}")

cohort_final.to_csv('/content/drive/MyDrive/mimic-iv/aortic_aneurysm_cohort_with_aki.csv', index=False)

In [ ]:
print(cohort_final.groupby('approach')['aki'].agg(['mean', 'count']))
print(cohort_final.groupby('location')['aki'].agg(['mean', 'count']))

In [ ]:
cohort_final.to_csv('/content/drive/MyDrive/mimic-iv/aortic_aneurysm_cohort_with_aki.csv', index=False)

In [ ]:
creat.to_csv('/content/drive/MyDrive/mimic-iv/aortic_aneurysm_creatinine_raw.csv', index=False)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import glob

BASE_PATH = '/content/drive/MyDrive'

cohort_final = pd.read_csv('/content/drive/MyDrive/mimic-iv/aortic_aneurysm_cohort_with_aki.csv', parse_dates=['admittime', 'dischtime'])
creat = pd.read_csv('/content/drive/MyDrive/mimic-iv/aortic_aneurysm_creatinine_raw.csv', parse_dates=['charttime'])
print(cohort_final.shape, creat.shape)

patients_path = glob.glob(f'{BASE_PATH}/**/patients.csv*', recursive=True)
print(patients_path)

In [ ]:
PATIENTS_PATH = '/content/drive/MyDrive/mimic-iv/hosp/patients.csv.gz'
patients = pd.read_csv(PATIENTS_PATH, usecols=['subject_id', 'anchor_age', 'anchor_year', 'gender'])

# Drop overlapping columns from cohort_final if they exist to avoid suffixing (_x, _y) during merge
cols_to_drop = [col for col in ['anchor_age', 'anchor_year', 'gender'] if col in cohort_final.columns]
if cols_to_drop:
    cohort_final = cohort_final.drop(columns=cols_to_drop)

cohort_final = cohort_final.merge(patients, on='subject_id', how='left')
cohort_final['age_at_admission'] = cohort_final['anchor_age'] + (cohort_final['admittime'].dt.year - cohort_final['anchor_year'])
print(cohort_final[['age_at_admission', 'gender']].describe(include='all'))

In [ ]:
cohort_final = cohort_final.drop(columns=['anchor_age', 'anchor_year', 'gender'], errors='ignore')
patients = pd.read_csv(PATIENTS_PATH, usecols=['subject_id', 'anchor_age', 'anchor_year', 'gender'])
cohort_final = cohort_final.merge(patients, on='subject_id', how='left')
cohort_final['age_at_admission'] = cohort_final['anchor_age'] + (cohort_final['admittime'].dt.year - cohort_final['anchor_year'])
print(cohort_final[['age_at_admission', 'gender']].describe(include='all'))

In [ ]:
diag_path = glob.glob(f'{BASE_PATH}/**/diagnoses_icd.csv*', recursive=True)
d_diag_path = glob.glob(f'{BASE_PATH}/**/d_icd_diagnoses.csv*', recursive=True)
print(diag_path, d_diag_path)

In [ ]:
!pip install comorbidipy --quiet
import comorbidipy
print(comorbidipy.__version__)

In [ ]:
import inspect
from comorbidipy import comorbidity
print(inspect.signature(comorbidity))
help(comorbidity)

In [ ]:
diag = pd.read_csv('/content/drive/MyDrive/mimic-iv/hosp/diagnoses_icd.csv.gz', dtype={'icd_code': str})
diag_cohort = diag[diag['hadm_id'].isin(cohort_final['hadm_id'])].copy()
print(diag_cohort['icd_version'].value_counts())

import polars as pl
from comorbidipy import comorbidity

diag9 = pl.from_pandas(diag_cohort[diag_cohort['icd_version'] == 9][['hadm_id', 'icd_code']])
diag10 = pl.from_pandas(diag_cohort[diag_cohort['icd_version'] == 10][['hadm_id', 'icd_code']])

cci9 = comorbidity(diag9, id_col='hadm_id', code_col='icd_code', icd='icd9')
cci10 = comorbidity(diag10, id_col='hadm_id', code_col='icd_code', icd='icd10')

cci_all = pl.concat([cci9, cci10], how='diagonal').to_pandas()
print(cci_all.shape)
print(cci_all['comorbidity_score'].describe())

In [ ]:
ditems_path = glob.glob(f'{BASE_PATH}/**/d_items.csv*', recursive=True)
print(ditems_path)

In [ ]:
d_items = pd.read_csv(ditems_path[0])
vaso_candidates = d_items[d_items['label'].str.contains('norepinephrine|epinephrine|vasopressin|phenylephrine|dopamine|dobutamine', case=False, na=False, regex=True)]
print(vaso_candidates[['itemid', 'label', 'linksto', 'category']].to_string(index=False))

In [ ]:
VASO_ITEMIDS = [221289, 221653, 221662, 221749, 221906, 222315, 229617, 229630, 229631, 229632]

inputevents_path = glob.glob(f'{BASE_PATH}/**/inputevents.csv*', recursive=True)
print(inputevents_path)

In [ ]:
inputevents = pd.read_csv(inputevents_path[0], usecols=['subject_id', 'hadm_id', 'itemid'])
vaso_hadm = inputevents[inputevents['itemid'].isin(VASO_ITEMIDS)]['hadm_id'].unique()

cohort_final['vasopressor'] = cohort_final['hadm_id'].isin(vaso_hadm).astype(int)
print(cohort_final['vasopressor'].value_counts())
print(f"Vasopressor use rate: {cohort_final['vasopressor'].mean():.1%}")

In [ ]:
cci_small = cci_all[['hadm_id', 'comorbidity_score']].rename(columns={'comorbidity_score': 'cci'})
cohort_model = cohort_final.merge(cci_small, on='hadm_id', how='left')
cohort_model['cci'] = cohort_model['cci'].fillna(0)  # safety net, though we confirmed 780/780 matched already

model_vars = ['hadm_id', 'aki', 'approach', 'location', 'anchor_year_group', 'age_at_admission', 'gender', 'cci', 'vasopressor']
cohort_model = cohort_model[model_vars].dropna()
print(f"Final modeling dataset: {len(cohort_model)} admissions")

cohort_model.to_csv('/content/drive/MyDrive/mimic-iv/aortic_aneurysm_final_modeling_dataset.csv', index=False)

In [ ]:
import statsmodels.formula.api as smf

model = smf.logit(
    'aki ~ C(approach, Treatment(reference="endovascular")) '
    '+ C(location, Treatment(reference="abdominal")) '
    '+ age_at_admission + C(gender) + cci + vasopressor '
    '+ C(anchor_year_group)',
    data=cohort_model
).fit()

print(model.summary())

import numpy as np
odds_ratios = pd.DataFrame({
    'OR': np.exp(model.params),
    'CI_lower': np.exp(model.conf_int()[0]),
    'CI_upper': np.exp(model.conf_int()[1]),
    'p_value': model.pvalues
})
print(odds_ratios)

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
import patsy

y, X = patsy.dmatrices(
    'aki ~ C(approach, Treatment(reference="endovascular")) '
    '+ C(location, Treatment(reference="abdominal")) '
    '+ age_at_admission + C(gender) + cci + vasopressor '
    '+ C(anchor_year_group)',
    data=cohort_model, return_type='dataframe'
)

vif = pd.DataFrame()
vif['variable'] = X.columns
vif['VIF'] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
print(vif.to_string(index=False))